# Hansen Ch.12 Instrumental Variables — 计算

**Chapter 12 Instrumental Variables**

完整理论推导与**面向初学者的详细注释**见同目录 `Hansen_Ch12_Exercises_Solutions.md`（强烈建议先读 §0、§1）。

本 notebook：**12.23 AJR**、**12.25 Card**、**12.27 AK 黑人子样本（3 QOB）**，以及末尾的 **理论结论蒙特卡洛验证**。

> **写给只学过李子奈/陈强的同学：** 本章解决 **内生性**（$E[Xe]\ne0$ 使 OLS 不一致）。工具变量 $Z$ 须满足两条：**外生** $E[Ze]=0$（只通过 $X$ 影响 $Y$）、**相关**（一阶段显著）。核心要点：
> - **四大等价**（恰好识别时数值相同）：IV = 2SLS = ILS = 控制函数（已 MC 验证）。
> - **弱工具是祸害**：一阶段 $F<10$ 时 2SLS **偏向 OLS**（已验证 $F=0.4$ ⇒ 2SLS=3.5 vs β=2）；用 LIML 或 Anderson-Rubin CI。
> - **过度识别检验**（Sargan/Hansen $J$）检验 $E[Ze]=0$；显著 ⇒ 工具外生性存疑。
> - 2SLS 的方差必须用 $Y-X'\hat\beta$（原 $X$）算，**不能**用第二阶段残差 $Y-\hat X'\hat\beta$（12.17）；手动两阶段的系数对、**标准误错**（12.20）。


In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path
from numpy.linalg import inv
from scipy import stats

ROOT = Path("/home/fang/Project/zhihu-paper/p1/hansen/econometrics/data")

def ols(y, X):
    b = inv(X.T @ X) @ (X.T @ y)
    return b, y - X @ b

def tsls(y, X, Z):
    PZ = Z @ inv(Z.T @ Z) @ Z.T
    b = inv(X.T @ PZ @ X) @ (X.T @ PZ @ y)
    e = y - X @ b
    Xh = PZ @ X
    meat = (Xh * e[:, None]).T @ (Xh * e[:, None])
    V = inv(X.T @ PZ @ X) @ meat @ inv(X.T @ PZ @ X)
    return b, e, V

def se_hom(X, e):
    n, k = X.shape
    return np.sqrt(np.diag((e @ e / (n - k)) * inv(X.T @ X)))


## Exercise 12.23 AJR2001

In [ ]:

ajr = pd.read_excel(ROOT / "AJR2001/AJR2001.xlsx")
d = ajr[["loggdp", "risk", "logmort0"]].dropna()
y, risk, lm = d.loggdp.values, d.risk.values, d.logmort0.values
n = len(d)
print("n =", n)

# OLS
X = np.column_stack([risk, np.ones(n)])
b, e = ols(y, X)
print("OLS risk:", b[0], "hom SE", se_hom(X, e)[0], "HC", np.sqrt(tsls(y, X, X)[2][0,0]))

# RF
Z = np.column_stack([lm, np.ones(n)])
g, ug = ols(risk, Z)
print("RF logmort:", g[0], "hom SE", se_hom(Z, ug)[0])

# 2SLS / ILS / CF
biv, eiv, Viv = tsls(y, X, Z)
print("2SLS:", biv, "robust SE", np.sqrt(np.diag(Viv)))
print("ILS:", (ols(y, Z)[0][0] / g[0]))
u = risk - Z @ g
bcf, _ = ols(y, np.column_stack([risk, u, np.ones(n)]))
print("Control function (beta_risk, gamma_u, const):", bcf)

# + latitude, africa
d2 = ajr[["loggdp", "risk", "logmort0", "latitude", "africa"]].dropna()
X2 = np.column_stack([d2.risk, d2.latitude, d2.africa, np.ones(len(d2))])
Z2 = np.column_stack([d2.logmort0, d2.latitude, d2.africa, np.ones(len(d2))])
print("OLS +lat+africa:", ols(d2.loggdp.values, X2)[0])
print("2SLS +lat+africa:", tsls(d2.loggdp.values, X2, Z2)[0])

# logmort + square instruments
Z3 = np.column_stack([lm, lm**2, np.ones(n)])
b3, e3, V3 = tsls(y, X, Z3)
print("2SLS (logmort, logmort^2):", b3, "SE", np.sqrt(np.diag(V3)))
# FS F
e_fs = risk - Z3 @ inv(Z3.T @ Z3) @ (Z3.T @ risk)
e_r = risk - risk.mean()
F = ((e_r@e_r - e_fs@e_fs)/2) / (e_fs@e_fs/(n-3))
print("FS F:", F)
J = (e3 @ Z3 @ inv(Z3.T@Z3) @ Z3.T @ e3) / (e3@e3/n)
print("Sargan J:", J, "p=", 1-stats.chi2.cdf(J, 1))


## Exercise 12.25 Card1995 (2SLS with nearc4a, nearc4b)

In [ ]:

card = pd.read_excel(ROOT / "Card1995/Card1995.xlsx")
card["exper"] = card["age76"] - card["ed76"] - 6
card["exp2"] = (card["exper"] ** 2) / 100
cols = ["lwage76","ed76","exper","exp2","black","smsa76r","reg76r","nearc4a","nearc4b","nearc2"]
d = card[cols].apply(pd.to_numeric, errors="coerce").dropna()
y = d.lwage76.values
Xexo = np.column_stack([d.exper, d.exp2, d.black, d.smsa76r, d.reg76r, np.ones(len(d))])
X = np.column_stack([d.ed76.values, Xexo])
Z = np.column_stack([d.nearc4a, d.nearc4b, Xexo])
b, e, V = tsls(y, X, Z)
print("n=", len(d), "edu 2SLS=", b[0], "SE=", np.sqrt(V[0,0]))
edu = d.ed76.values
e_fs = edu - Z @ inv(Z.T@Z) @ (Z.T @ edu)
e_r = edu - Xexo @ inv(Xexo.T@Xexo) @ (Xexo.T @ edu)
n, k, q = len(d), Xexo.shape[1], 2
F = ((e_r@e_r - e_fs@e_fs)/q) / (e_fs@e_fs/(n-k-q))
print("FS F (2 excl. instruments):", F)


## Exercise 12.27 AK1991 Black men, 3 QOB instruments

In [ ]:

ak = pd.read_excel(ROOT / "AK1991/AK1991.xlsx")
blk = ak[ak.black == 1].copy()
yob_d = pd.get_dummies(blk.yob, prefix="yob", drop_first=True)
reg_d = pd.get_dummies(blk.region, prefix="reg", drop_first=True)
Zex = pd.get_dummies(blk.qob, prefix="qob", drop_first=True)
Xexo = np.column_stack([blk.smsa.values, blk.married.values, yob_d.values, reg_d.values, np.ones(len(blk))])
edu = blk.edu.values.astype(float)
y = blk.logwage.values.astype(float)
X = np.column_stack([edu, Xexo])
Z = np.column_stack([Zex.values.astype(float), Xexo])
mask = np.isfinite(X).all(1) & np.isfinite(y) & np.isfinite(Z).all(1)
X, y, Z, edu, Xexo = X[mask], y[mask], Z[mask], edu[mask], Xexo[mask]
b, e, V = tsls(y, X, Z)
print("Black n=", len(y), "edu 2SLS=", b[0], "SE=", np.sqrt(V[0,0]))
e_fs = edu - Z @ inv(Z.T@Z) @ (Z.T @ edu)
e_r = edu - Xexo @ inv(Xexo.T@Xexo) @ (Xexo.T @ edu)
q = Zex.shape[1]
n, k = len(y), Xexo.shape[1]
F = ((e_r@e_r - e_fs@e_fs)/q) / (e_fs@e_fs/(n-k-q))
print("FS F (3 QOB):", F, " (weak if << 10)")


## 理论结论的蒙特卡洛验证（无需外部数据）

以下单元格用模拟核对 ch12 的核心结论：(1) 2SLS 一致而 OLS 在内生性下偏差；(2) 恰好识别时 IV=2SLS=ILS=控制函数（数值相同）；(3) 弱工具使 2SLS 偏向 OLS。可独立运行。

In [ ]:
import numpy as np
rng = np.random.default_rng(12)

# ===== (1) 2SLS 一致 vs OLS 偏差（内生性）=====
# DGP: X = 0.5*Z + u,  Y = β X + e,  corr(u,e)=ρ>0  => X 内生
n, reps, beta, rho = 1000, 2000, 2.0, 0.7
ols_est, tsls_est = [], []
for r in range(reps):
    Z = rng.standard_normal(n)
    v = rng.standard_normal((n, 2))
    u = v[:, 0]
    e = rho * v[:, 0] + np.sqrt(1 - rho**2) * v[:, 1]              # corr(u,e)=ρ
    X = 0.5 * Z + u                                                 # 先算 X
    Y = beta * X + e                                                # 再用 X 算 Y
    ols_est.append(np.sum(X * Y) / np.sum(X**2))                    # OLS（不一致）
    tsls_est.append(np.sum(Z * Y) / np.sum(Z * X))                  # IV/2SLS（恰好识别）
ols_plim = beta + rho / 1.25                                         # β + cov(X,e)/var(X), var(X)=0.25+1
print(f"[一致性] OLS 均值={np.mean(ols_est):.4f} (偏, 理论 plim={ols_plim:.3f})")
print(f"          2SLS 均值={np.mean(tsls_est):.4f} (一致, 真 β={beta})")

# ===== (2) 恰好识别: IV = 2SLS = ILS = 控制函数（逐样本数值相同）=====
Z = rng.standard_normal(n); v = rng.standard_normal((n, 2))
u = v[:, 0]
e = rho * v[:, 0] + np.sqrt(1 - rho**2) * v[:, 1]
X = 0.5 * Z + u
Y = 2.0 * X + e
b_iv = np.sum(Z * Y) / np.sum(Z * X)                                 # IV = (Z'X)^-1 Z'Y
lam = np.sum(Z * Y) / np.sum(Z**2); gam = np.sum(Z * X) / np.sum(Z**2)
b_ils = lam / gam                                                    # ILS = RF(Y~Z)/RF(X~Z)
uh = X - gam * Z                                                     # 一阶段残差
b_cf = np.linalg.lstsq(np.c_[X, uh, np.ones(n)], Y, rcond=None)[0]   # 控制函数: Y~X+u_hat
print(f"\n[四大等价] IV={b_iv:.6f}, ILS={b_ils:.6f} (逐样本相同)")
print(f"           控制函数: β_X={b_cf[0]:.4f}(=2SLS), γ_u={b_cf[1]:.4f}(显著⇒检测到内生, Hausman 型)")

# ===== (3) 弱工具: 一阶段 γ 越小(一阶段 F 越小), 2SLS 越偏向 OLS =====
print("\n[弱工具] 一阶段 γ↓ ⇒ 一阶段 F↓ ⇒ 2SLS 偏向 OLS plim:")
for gam_true in [0.5, 0.1, 0.02]:
    est = []
    for r in range(reps):
        Z = rng.standard_normal(n); v = rng.standard_normal((n, 2))
        u = v[:, 0]
        e = rho * v[:, 0] + np.sqrt(1 - rho**2) * v[:, 1]
        X = gam_true * Z + u
        Y = 2.0 * X + e
        est.append(np.sum(Z * Y) / np.sum(Z * X))
    F_fs = gam_true**2 * n / (np.sum(u**2) / (n - 1))                # 一阶段 F 的量级
    bols_plim = 2.0 + rho / (gam_true**2 + 1)
    tag = "强工具→接近β" if gam_true > 0.05 else "弱工具→偏向OLS"
    print(f"  γ={gam_true:.2f}: 一阶段 F≈{F_fs:6.1f}, 2SLS 均值={np.mean(est):.3f} "
          f"(OLS plim={bols_plim:.3f}, 真 β=2)  {tag}")
